In [1]:
import os
import pandas as pd
import pyarrow.feather as feather
import torch
from collections import defaultdict

def clean_UserID(uid):
    """
    UserID 값이 문자열인 경우, "::"가 포함되어 있다면 첫 부분만 반환.
    그 외에는 문자열로 변환하여 반환.
    """
    if isinstance(uid, str):
        if "::" in uid:
            return uid.split("::")[0]
        return uid
    return str(uid)

def read_feather_with_type(filepath):
    """
    Feather 파일을 읽고, 파일 내 'Vector'가 포함된 컬럼을 리스트 타입으로 변환합니다.
    예: "URE_Vector", "UR_Vector", "UI_Vector"
    또한, UserID 컬럼을 문자열(str)로 통일하고, 형식이 잘못된 경우 정리합니다.
    """
    df = feather.read_feather(filepath)
    if "UserID" in df.columns:
        df["UserID"] = df["UserID"].apply(clean_UserID).astype(str)
    
    # 'Vector'라는 단어가 포함된 컬럼 찾기 (여러 개 있을 경우 첫 번째만 사용)
    vector_cols = [col for col in df.columns if "Vector" in col]
    if not vector_cols:
        raise ValueError(f"'{filepath}'에 'Vector'가 포함된 컬럼이 없습니다.")
    vector_col = vector_cols[0]
    df[vector_col] = df[vector_col].apply(lambda x: x if isinstance(x, list) else list(x))
    return df, vector_col

def concat_vectors_with_torch(base_dir, folders=None):
    """
    base_dir 내의 지정된 폴더들에서 Feather 파일을 읽어 UserID 기준으로 full outer merge 후,
    각 행의 Vector 컬럼들을 torch.cat으로 연결하여 'concatenated_vector' 컬럼 생성.
    """
    if folders is None:
        folders = ['URE_output', 'UI(doc2vec)(2)_output', 'UR_output']

    category_dfs = defaultdict(list)
    
    for folder in folders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.exists(folder_path):
            continue
        
        for file in os.listdir(folder_path):
            if file.endswith('.feather'):
                category = file.split('_')[0]  # 'category1_UI.feather' → 'category1'
                filepath = os.path.join(folder_path, file)
                df, vector_col = read_feather_with_type(filepath)
                df = df[['UserID', vector_col]].rename(columns={vector_col: f'{folder}_{vector_col}'})
                category_dfs[category].append(df)
    
    result = {}
    for category, dfs in category_dfs.items():
        if not dfs:
            continue
        # UserID 기준 full outer join
        merged_df = dfs[0]
        for df in dfs[1:]:
            merged_df = pd.merge(merged_df, df, on='UserID', how='outer')
        
        # 벡터 컬럼 모으기
        vector_cols = [col for col in merged_df.columns if col != 'UserID']
        
        # 각 행마다 torch.cat
        def concat_row(row):
            vectors = []
            for col in vector_cols:
                v = row[col]
                if isinstance(v, list):
                    vectors.append(torch.tensor(v))
            if vectors:
                return torch.cat(vectors).tolist()
            else:
                return []
        
        merged_df['concatenated_vector'] = merged_df.apply(concat_row, axis=1)
        result[category] = merged_df[['UserID', 'concatenated_vector']]
    
    return result

def save_results(results, base_dir):
    """
    results 딕셔너리에 있는 각 카테고리의 DataFrame을
    base_dir/concatenate 폴더에 카테고리별 Feather 파일로 저장.
    """
    output_folder = os.path.join(base_dir, "concatenate(doc2vec)(2)")
    os.makedirs(output_folder, exist_ok=True)
    
    for category, df in results.items():
        out_filepath = os.path.join(output_folder, f"{category}.feather")
        feather.write_feather(df, out_filepath)
        print(f"Saved {category} to {out_filepath}")

# 사용 예시
if __name__ == "__main__":
    base_directory = "../output"
    merged_data = concat_vectors_with_torch(base_directory)
    
    # 결과를 출력
    for category, df in merged_data.items():
        print(f"=== {category} ===")
        print(df.head())
    
    # 각 카테고리별 결과를 output/concatenate 폴더에 저장
    save_results(merged_data, base_directory)


=== Adult Products ===
    UserID                                concatenated_vector
0    11996  [-0.2758602797985077, 0.01675277203321457, -0....
1     2815  [-0.2635284662246704, -0.027993008494377136, -...
2  5001156  [-0.3294433355331421, 0.10732106119394302, -0....
3  5002475  [-0.3380291759967804, -0.018190458416938782, -...
4  5016738  [-0.3468416929244995, 0.09743359684944153, -0....
=== Beauty ===
  UserID                                concatenated_vector
0  10172  [0.08522136509418488, 0.2976636290550232, 0.46...
1  10203  [0.09062256664037704, 0.34977588057518005, 0.2...
2  10212  [0.05668530613183975, 0.286480575799942, 0.365...
3  10234  [0.2765531837940216, -0.16571319103240967, -0....
4  10269  [-0.08141428977251053, 0.6790059804916382, 0.6...
=== Books ===
  UserID                                concatenated_vector
0  10203  [-1.2007917165756226, 0.4883524775505066, -0.7...
1  10212  [-0.3126230239868164, 0.013458602130413055, -0...
2  10218  [-0.532976508140564, 0.117

In [7]:
out_filepath = '../output/concatenate/concatenate(doc2vec)(200dim)/Adult Products.feather'
df = pd.read_feather(out_filepath)
print(out_filepath)
df
# 랜덤으로 1개의 행을 선택
random_row = df.sample(n=1)

# 선택된 행에서 'Vector' 열만 출력 (Vector_UR과 Vector_UI)
vector_columns = [col for col in df.columns if "concatenated_vector" in col]
random_vector = random_row[vector_columns]

print(random_vector.to_string(index=False))


../output/concatenate/concatenate(doc2vec)(200dim)/Adult Products.feather
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              